# Data Mining Lab 1: Spark Data Lake Preprocessing

**Dataset:** `sales.csv` | **Environment:** Google Colab / PySpark 4.0.1

**Student name:** Priyanshu Agarwal  **Roll number:** LIT2023022

This notebook preserves the original CSV in the **Bronze** layer, uses Spark for data inspection and preprocessing, and saves the processed data in the **Silver** layer. All discrepancy counts are calculated using Spark operations. Run the notebook sequentially from top to bottom using **Runtime > Run all** in Colab. No **Gold** layer or detailed sales analysis is required.


## 1. Install PySpark and create a SparkSession

Colab supplies Java. For another Jupyter environment, use Python 3.9+ and
Java 17+ as required by [PySpark 4.0.1](https://spark.apache.org/docs/4.0.1/api/python/getting_started/install.html).
The version is pinned so the safe parsing functions used below are available.


In [1]:
%pip -q install "pyspark==4.0.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 1.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import shutil

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("DataMiningLab1")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


Spark version: 4.0.1


## 2. Create Bronze and Silver folders

Download the original bytes directly into Bronze. Do not open and resave the
CSV with a data library: doing so could alter its formatting. Verify its SHA-256
checksum now and again after preprocessing.

```text
data_lake/
  bronze/sales.csv
  silver/preprocessed_sales/       # Spark CSV output directory
silver/sales_silver.csv             # Same complete output, convenient to submit
```


In [3]:
ROOT = Path.cwd()
BRONZE = ROOT / "data_lake" / "bronze"
SILVER = ROOT / "data_lake" / "silver" / "preprocessed_sales"
SUBMISSION_SILVER = ROOT / "silver"
BRONZE.mkdir(parents=True, exist_ok=True)
SILVER.parent.mkdir(parents=True, exist_ok=True)
SUBMISSION_SILVER.mkdir(parents=True, exist_ok=True)

SOURCE_REVISION = "81539d8aee71b8ce4c428365cea26e6b272723c6"
SOURCE_URL = (
    "https://raw.githubusercontent.com/necromancersanju-netizen/"
    f"data-mining-data-lake-assignment/{SOURCE_REVISION}/dataset/sales.csv"
)
EXPECTED_SHA256 = "1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8"
bronze_file = BRONZE / "sales.csv"

if not bronze_file.exists():
    with urlopen(SOURCE_URL, timeout=60) as response:
        source_bytes = response.read()
    assert hashlib.sha256(source_bytes).hexdigest() == EXPECTED_SHA256
    bronze_file.write_bytes(source_bytes)

bronze_hash_before = hashlib.sha256(bronze_file.read_bytes()).hexdigest()
assert bronze_hash_before == EXPECTED_SHA256, "Existing Bronze file differs from the source."
print("Source:", SOURCE_URL)
print("Bronze SHA-256:", bronze_hash_before)


Source: https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/81539d8aee71b8ce4c428365cea26e6b272723c6/dataset/sales.csv
Bronze SHA-256: 1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8


## 3. Load and inspect the raw data

Read all fields as strings first. This preserves invalid numeric and date text
for inspection instead of silently turning it into null during schema inference.
Spark reads empty CSV fields as null. Extra spaces are preserved at this stage.


In [4]:
raw_df = (
    spark.read.option("header", True)
    .option("inferSchema", False)
    .option("ignoreLeadingWhiteSpace", False)
    .option("ignoreTrailingWhiteSpace", False)
    .option("mode", "FAILFAST")
    .csv(str(bronze_file))
    .cache()
)
original_columns = raw_df.columns
before_count = raw_df.count()
raw_df.printSchema()
raw_df.show(8, truncate=False)
print("Bronze records:", before_count)


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product      |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+--

In [5]:
# Remove only full-row duplicates, before modifying any values.
dedup_df = raw_df.dropDuplicates().cache()
dedup_count = dedup_df.count()
duplicate_count = before_count - dedup_count
print("Exact duplicate rows to remove:", duplicate_count)
raw_df.groupBy(*original_columns).count().filter(F.col("count") > 1).show(5, truncate=False)

# Repeated order IDs are not automatically full-row duplicates.
repeated_ids = dedup_df.groupBy("order_id").count().filter(F.col("count") > 1)
print("Order IDs still appearing in multiple different rows:", repeated_ids.count())
repeated_ids.orderBy("order_id").show(truncate=False)


Exact duplicate rows to remove: 10
+--------+-----------+--------------+---------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name |product        |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+--------------+---------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100444  |C0090      |Abhinav Pillai|Novel          |Books    |8       |530       |15              |Credit Card     |Mumbai    |Maharashtra   |2025-09-11|Delivered   |2    |
|100773  |C0315      |Meera Malhotra|Study Table    |Furniture|7       |11310     |40              |Debit Card      |Pune      |Maharashtra   |2026-02-25|Delivered   |2    |
|100297  |C0149      |Manish Pillai |Dry Fruits Pack|Grocery  |7       |780       |35          

## 4. Identify missing values and text inconsistencies

Missing means null or a whitespace-only field. Standardize names, categories,
cities, states, and status to title case. Preserve `UPI`, `Cash on Delivery`,
and existing product spellings such as `USB-C Hub`, `T-Shirt`, and `Tea 1kg`.
Product labels need trimming only in this dataset. IDs remain identifiers.


In [6]:
text_columns = [
    "order_id", "customer_id", "customer_name", "product", "category",
    "payment_method", "city", "state", "order_status"
]
description_columns = [c for c in text_columns if c not in ("order_id", "customer_id")]
title_columns = {"customer_name", "category", "city", "state", "order_status"}

def is_missing(column):
    value = F.col(column)
    return value.isNull() | (F.trim(value) == "")

def normalize_text(column):
    value = F.trim(F.col(column))
    if column in title_columns:
        return F.initcap(value)
    if column == "customer_id":
        return F.upper(value)
    if column == "payment_method":
        return (
            F.when(F.upper(value) == "UPI", F.lit("UPI"))
            .when(F.lower(value) == "cash on delivery", F.lit("Cash on Delivery"))
            .otherwise(F.initcap(value))
        )
    return value

def count_conditions(frame, conditions):
    # Spark computes every count. Only the small summary is collected.
    return frame.agg(*[
        F.count(F.when(condition, 1)).alias(label)
        for label, condition in conditions.items()
    ]).first().asDict()

def show_counts(counts):
    spark.createDataFrame(list(counts.items()), ["discrepancy", "count"]).show(100, truncate=False)

missing_before = count_conditions(raw_df, {c: is_missing(c) for c in original_columns})
print("Missing values in Bronze:")
show_counts(missing_before)

text_changes_before = count_conditions(raw_df, {
    c: (~is_missing(c)) & (~F.col(c).eqNullSafe(normalize_text(c)))
    for c in text_columns
})
print("Values needing trimming or capitalization standardization:")
show_counts(text_changes_before)

for c in ["category", "city"]:
    print("Observed variants in", c)
    raw_df.filter(~F.col(c).eqNullSafe(normalize_text(c))).select(c).distinct().show(truncate=False)


Missing values in Bronze:
+----------------+-----+
|discrepancy     |count|
+----------------+-----+
|order_id        |0    |
|customer_id     |0    |
|customer_name   |10   |
|product         |0    |
|category        |0    |
|quantity        |0    |
|unit_price      |0    |
|discount_percent|0    |
|payment_method  |8    |
|city            |8    |
|state           |0    |
|order_date      |0    |
|order_status    |0    |
+----------------+-----+

Values needing trimming or capitalization standardization:
+--------------+-----+
|discrepancy   |count|
+--------------+-----+
|order_id      |0    |
|customer_id   |0    |
|customer_name |3    |
|product       |0    |
|category      |6    |
|payment_method|0    |
|city          |9    |
|state         |0    |
|order_status  |0    |
+--------------+-----+

Observed variants in category
+-----------+
|category   |
+-----------+
|ELECTRONICS|
|CLOTHING   |
|sports     |
|beauty     |
|SPORTS     |
+-----------+

Observed variants in city
+-----

## 5. Identify invalid numbers and dates

The rules for this lab are:

| Field | Accepted values | Handling of an invalid or missing value |
|---|---|---|
| quantity | Positive whole number, representable as a Spark integer | Null, retain the row |
| unit_price | Positive price, representable as `decimal(12,2)` | Null, retain the row |
| discount_percent | Between 0 and 100 inclusive | Null, retain the row |
| order_date | A real calendar date in one of the formats below | Null, retain the row |

Zero quantity and zero price are treated as invalid for this sales lab. A real
business that permits free items would need a different price rule. We do not
impose an arbitrary upper outlier cutoff, fill a missing discount with zero, or
invent transaction values from averages.

Accepted dates: `yyyy-MM-dd`, `yyyy/MM/dd`, `dd/MM/yyyy`, and `MM-dd-yyyy`.
Slash-separated day-first dates are interpreted as `dd/MM/yyyy`; hyphenated
month-first dates are interpreted as `MM-dd-yyyy`. These are explicit parsing
choices, not guesses for each row. Invalid calendar dates are never rolled over.
`try_to_timestamp` and `try_cast` return null when parsing fails.


In [7]:
date_formats = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "MM-dd-yyyy"]

def parsed_date(column):
    return F.coalesce(*[
        F.try_to_timestamp(F.trim(F.col(column)), F.lit(fmt))
        for fmt in date_formats
    ]).cast("date")

def with_parsed_fields(frame):
    return (
        frame.withColumn("_quantity", F.trim(F.col("quantity")).try_cast("double"))
        .withColumn("_unit_price", F.trim(F.col("unit_price")).try_cast("decimal(12,2)"))
        .withColumn("_discount", F.trim(F.col("discount_percent")).try_cast("double"))
        .withColumn("_date", parsed_date("order_date"))
    )

valid_quantity = (
    F.col("_quantity").isNotNull() & (~F.isnan("_quantity"))
    & (F.col("_quantity") > 0) & (F.col("_quantity") <= 2147483647)
    & (F.col("_quantity") == F.floor("_quantity"))
)
valid_price = F.col("_unit_price").isNotNull() & (F.col("_unit_price") > 0)
valid_discount = (
    F.col("_discount").isNotNull() & (~F.isnan("_discount"))
    & F.col("_discount").between(0, 100)
)

invalid_conditions = {
    "invalid_or_missing_quantity": ~valid_quantity,
    "invalid_or_missing_unit_price": ~valid_price,
    "invalid_or_missing_discount_percent": ~valid_discount,
    "invalid_or_missing_order_date": F.col("_date").isNull(),
}
raw_parsed = with_parsed_fields(raw_df)
numeric_date_before = count_conditions(raw_parsed, invalid_conditions)
print("Invalid/missing numbers and dates in Bronze:")
show_counts(numeric_date_before)

date_format_counts = count_conditions(raw_parsed, {
    "non_ISO_date_strings": (~is_missing("order_date")) &
        (~F.trim(F.col("order_date")).rlike(r"^\d{4}-\d{2}-\d{2}$")),
    "valid_dates_needing_reformatting": F.col("_date").isNotNull() &
        (F.trim(F.col("order_date")) != F.date_format("_date", "yyyy-MM-dd")),
})
show_counts(date_format_counts)
raw_parsed.filter(
    F.col("_date").isNull() |
    (F.trim(F.col("order_date")) != F.date_format("_date", "yyyy-MM-dd"))
).select("order_id", "order_date", "_date").show(truncate=False)


Invalid/missing numbers and dates in Bronze:
+-----------------------------------+-----+
|discrepancy                        |count|
+-----------------------------------+-----+
|invalid_or_missing_quantity        |12   |
|invalid_or_missing_unit_price      |8    |
|invalid_or_missing_discount_percent|8    |
|invalid_or_missing_order_date      |4    |
+-----------------------------------+-----+

+--------------------------------+-----+
|discrepancy                     |count|
+--------------------------------+-----+
|non_ISO_date_strings            |4    |
|valid_dates_needing_reformatting|2    |
+--------------------------------+-----+

+--------+----------+----------+
|order_id|order_date|_date     |
+--------+----------+----------+
|100097  |31/02/2026|NULL      |
|100358  |2025/07/14|2025-07-14|
|100448  |not-a-date|NULL      |
|100634  |2025-02-30|NULL      |
|100637  |2026-13-05|NULL      |
|100969  |09-15-2026|2026-09-15|
+--------+----------+----------+



## 6. Basic preprocessing

Start from the deduplicated records. Fill missing descriptive text with
`Unknown`. Set invalid numerical values and invalid dates to null, with a
`data_quality_issues` column to explain those gaps. Bronze retains the original
values for later investigation. Keep conflicting records that share an order ID
and all valid unusual values: only exact duplicate rows are removed.

Text-format corrections are counted above. The quality flag records unresolved
missing/invalid information, not harmless capitalization changes.


In [8]:
working = with_parsed_fields(dedup_df)
issue_conditions = {
    **{f"missing_{c}": is_missing(c) for c in description_columns},
    **invalid_conditions,
}
issues_after_dedup = count_conditions(working, issue_conditions)
print("Issues handled in retained records:")
show_counts(issues_after_dedup)

working = working.withColumn("data_quality_issues", F.concat_ws("; ", *[
    F.when(condition, F.lit(label)) for label, condition in issue_conditions.items()
]))
working = working.withColumn(
    "data_quality_issues",
    F.when(F.col("data_quality_issues") == "", F.lit("none"))
    .otherwise(F.col("data_quality_issues"))
)

for c in text_columns:
    working = working.withColumn(
        c,
        F.when(is_missing(c), F.lit("Unknown") if c in description_columns else F.lit(None))
        .otherwise(normalize_text(c))
    )

silver_df = (
    working.withColumn("quantity", F.when(valid_quantity, F.col("_quantity").cast("int")))
    .withColumn("unit_price", F.when(valid_price, F.col("_unit_price")))
    .withColumn("discount_percent", F.when(valid_discount, F.col("_discount").cast("decimal(5,2)")))
    .withColumn("order_date", F.col("_date"))
    .select(*original_columns, "data_quality_issues")
    .cache()
)
after_count = silver_df.count()
silver_df.printSchema()
silver_df.orderBy("order_id").show(8, truncate=False)
print("Examples retaining useful records with a quality flag:")
silver_df.filter(F.col("data_quality_issues") != "none").select(
    "order_id", "quantity", "unit_price", "discount_percent", "order_date", "data_quality_issues"
).orderBy("order_id").show(10, truncate=False)


Issues handled in retained records:
+-----------------------------------+-----+
|discrepancy                        |count|
+-----------------------------------+-----+
|missing_customer_name              |10   |
|missing_product                    |0    |
|missing_category                   |0    |
|missing_payment_method             |8    |
|missing_city                       |8    |
|missing_state                      |0    |
|missing_order_status               |0    |
|invalid_or_missing_quantity        |12   |
|invalid_or_missing_unit_price      |8    |
|invalid_or_missing_discount_percent|8    |
|invalid_or_missing_order_date      |4    |
+-----------------------------------+-----+

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 

## 7. Save Silver and prepare the submission CSV

Spark writes CSV data to a directory of part files. Because this lab is small,
`coalesce(1)` creates one part file. Copy that complete file to the requested
`silver/sales_silver.csv` submission path. This is a full output, not a sample.
For large datasets, retain multiple partitions instead of using `coalesce(1)`.

Dates are saved as `yyyy-MM-dd`; intentional nulls are empty CSV fields.


In [9]:
(
    silver_df.orderBy(*original_columns).coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .option("dateFormat", "yyyy-MM-dd")
    .option("nullValue", "")
    .csv(str(SILVER))
)
part_files = list(SILVER.glob("part-*.csv"))
assert len(part_files) == 1, "Expected one data part for this small lab."
submission_csv = SUBMISSION_SILVER / "sales_silver.csv"
shutil.copyfile(part_files[0], submission_csv)
print("Spark output:", SILVER)
print("Submission CSV:", submission_csv)


Spark output: /content/data_lake/silver/preprocessed_sales
Submission CSV: /content/silver/sales_silver.csv


## 8. Read back and verify

Read the Silver directory with the same schema so numeric and date columns keep
their types. Check the row counts, full row contents (including duplicate
multiplicity), numeric validity, and the Bronze checksum. Remaining null counts
are shown explicitly: they represent invalid or missing values retained for
future correction.


In [10]:
readback_df = (
    spark.read.option("header", True)
    .option("dateFormat", "yyyy-MM-dd")
    .option("mode", "FAILFAST")
    .schema(silver_df.schema)
    .csv(str(SILVER))
)
readback_count = readback_df.count()
bronze_hash_after = hashlib.sha256(bronze_file.read_bytes()).hexdigest()

assert after_count == dedup_count
assert before_count - after_count == duplicate_count
assert readback_count == after_count
assert bronze_hash_before == bronze_hash_after == EXPECTED_SHA256
assert readback_df.exceptAll(silver_df).limit(1).count() == 0
assert silver_df.exceptAll(readback_df).limit(1).count() == 0
assert readback_df.filter(F.col("quantity").isNotNull() & (F.col("quantity") <= 0)).count() == 0
assert readback_df.filter(F.col("unit_price").isNotNull() & (F.col("unit_price") <= 0)).count() == 0
assert readback_df.filter(
    F.col("discount_percent").isNotNull() & (~F.col("discount_percent").between(0, 100))
).count() == 0

verification_counts = {
    "Bronze rows before preprocessing": before_count,
    "Exact duplicate rows removed": duplicate_count,
    "Silver rows after preprocessing": after_count,
    "Silver rows read back": readback_count,
    "Retained rows with a quality flag": silver_df.filter(F.col("data_quality_issues") != "none").count(),
}
show_counts(verification_counts)
print("Remaining nulls in Silver:")
remaining_nulls = count_conditions(readback_df, {c: F.col(c).isNull() for c in original_columns})
show_counts(remaining_nulls)
readback_df.orderBy("order_id").show(8, truncate=False)
print("Verification passed. Bronze is byte-for-byte unchanged.")


+---------------------------------+-----+
|discrepancy                      |count|
+---------------------------------+-----+
|Bronze rows before preprocessing |1000 |
|Exact duplicate rows removed     |10   |
|Silver rows after preprocessing  |990  |
|Silver rows read back            |990  |
|Retained rows with a quality flag|58   |
+---------------------------------+-----+

Remaining nulls in Silver:
+----------------+-----+
|discrepancy     |count|
+----------------+-----+
|order_id        |0    |
|customer_id     |0    |
|customer_name   |0    |
|product         |0    |
|category        |0    |
|quantity        |12   |
|unit_price      |8    |
|discount_percent|8    |
|payment_method  |0    |
|city            |0    |
|state           |0    |
|order_date      |4    |
|order_status    |0    |
+----------------+-----+

+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+--

## 9. What Was Performed

1. Kept the original file unchanged in the **Bronze** layer and verified its checksum.

2. Used Spark to examine the schema, sample records, row counts, duplicate rows, missing values, text variations, invalid numeric values, and date inconsistencies.

3. Removed exact duplicate records before performing any transformations.

4. Trimmed text fields and standardized obvious capitalization while retaining acronyms and original product spellings.

5. Replaced missing descriptive text fields with `Unknown`.

6. Converted invalid quantity, unit price, discount, and date values to `null`, while recording the corresponding reason in `data_quality_issues` instead of removing the affected rows.

7. Standardized valid date values and saved the resulting typed dataset as CSV in the **Silver** layer.

8. Loaded the Silver data back, compared complete row contents, and verified the row counts before and after processing.

**Understanding the Result:**
The **Bronze** layer represents the original source data, while **Silver** provides a cleaned and usable version with basic formatting and validation. A `null` indicates that the actual value is unknown; inserting an assumed value would conceal the data-quality issue. Similarly, an order ID by itself is not sufficient justification for removing a record when its other fields contain different information.
